In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("City analysis") \
    .master("local[*]") \
    .getOrCreate()


In [2]:
data  =  spark.read.csv('./stocks_price_final.csv', sep= ',', header = True)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/e:/DE-project-using-Docker-apache-kafka-spark-and-airflow/test/stocks_price_final.csv.

In [ ]:
data.head(3)

[Row(_c0='1', symbol='TXG', date='2019-09-12', open='54', high='58', low='51', close='52.75', volume='7326300', adjusted='52.75', market.cap='$9.31B', sector='Capital Goods', industry='Biotechnology: Laboratory Analytical Instruments', exchange='NASDAQ'),
 Row(_c0='2', symbol='TXG', date='2019-09-13', open='52.75', high='54.355', low='49.150002', close='52.27', volume='1025200', adjusted='52.27', market.cap='$9.31B', sector='Capital Goods', industry='Biotechnology: Laboratory Analytical Instruments', exchange='NASDAQ'),
 Row(_c0='3', symbol='TXG', date='2019-09-16', open='52.450001', high='56', low='52.009998', close='55.200001', volume='269900', adjusted='55.200001', market.cap='$9.31B', sector='Capital Goods', industry='Biotechnology: Laboratory Analytical Instruments', exchange='NASDAQ')]

In [4]:
data.printSchema()


root
 |-- _c0: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- date: string (nullable = true)
 |-- open: string (nullable = true)
 |-- high: string (nullable = true)
 |-- low: string (nullable = true)
 |-- close: string (nullable = true)
 |-- volume: string (nullable = true)
 |-- adjusted: string (nullable = true)
 |-- market.cap: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- exchange: string (nullable = true)



In [5]:
from pyspark.sql.types import *

data_schema = [
    StructField("_c0", IntegerType(), True),
    StructField("symbol", StringType(), True),
    StructField("open", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("volume", IntegerType(), True),
    StructField("adjusted", DoubleType(), True),
    StructField("market.cap", StringType(), True),
    StructField("sector", StringType(), True),
    StructField("industry", StringType(), True),
    StructField("exchange", StringType(), True),
]

final_struct = StructType(fields=data_schema)
data = spark.read.csv(
    "./stocks_price_final.csv", sep=",", header=True, schema=final_struct
)

data.printSchema()


root
 |-- _c0: integer (nullable = true)
 |-- symbol: string (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: integer (nullable = true)
 |-- adjusted: double (nullable = true)
 |-- market.cap: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- exchange: string (nullable = true)



In [6]:
data.head()


Row(_c0=1, symbol='TXG', open=None, high=54.0, low=58.0, close=51.0, volume=None, adjusted=7326300.0, market.cap='52.75', sector='$9.31B', industry='Capital Goods', exchange='Biotechnology: Laboratory Analytical Instruments')

In [7]:
data.show()

+---+------+----+---------+---------+---------+------+---------+----------+------+-------------+--------------------+
|_c0|symbol|open|     high|      low|    close|volume| adjusted|market.cap|sector|     industry|            exchange|
+---+------+----+---------+---------+---------+------+---------+----------+------+-------------+--------------------+
|  1|   TXG|null|     54.0|     58.0|     51.0|  null|7326300.0|     52.75|$9.31B|Capital Goods|Biotechnology: La...|
|  2|   TXG|null|    52.75|   54.355|49.150002|  null|1025200.0|     52.27|$9.31B|Capital Goods|Biotechnology: La...|
|  3|   TXG|null|52.450001|     56.0|52.009998|  null| 269900.0| 55.200001|$9.31B|Capital Goods|Biotechnology: La...|
|  4|   TXG|null|56.209999|60.900002|   55.423|  null| 602800.0| 56.779999|$9.31B|Capital Goods|Biotechnology: La...|
|  5|   TXG|null|56.849998|    62.27|55.650002|    62|1589600.0|        62|$9.31B|Capital Goods|Biotechnology: La...|
|  6|   TXG|null|62.810001|   63.375|61.029999|  null| 4

In [8]:
data.describe(['high', 'low']).show()

+-------+------------------+------------------+
|summary|              high|               low|
+-------+------------------+------------------+
|  count|           1726301|           1726301|
|   mean|15070.071703341047|15555.067268137085|
| stddev|1111821.8002863186|1148247.1953514975|
|    min|             0.072|             0.078|
|    max|      1.60168176E8|      1.61601456E8|
+-------+------------------+------------------+



In [9]:
data.select("high", "low", "volume").summary("count", "min", "25%", "75%", "max").show()

+-------+------------+------------+---------+
|summary|        high|         low|   volume|
+-------+------------+------------+---------+
|  count|     1726301|     1726301|    40333|
|    min|       0.072|       0.078|        1|
|    25%|        7.03|        7.24|        9|
|    75%|   44.700001|       45.41|       56|
|    max|1.60168176E8|1.61601456E8|158376592|
+-------+------------+------------+---------+



## MISSING DATA PROCESS

In [10]:
data = data.na.fill(0)


## QUERY

In [11]:
# SELECT

## Selecting Single Column

data.select('sector').show(5)

## Selecting Multiple columns

data.select(['open', 'close', 'adjusted']).show(5)


+------+
|sector|
+------+
|$9.31B|
|$9.31B|
|$9.31B|
|$9.31B|
|$9.31B|
+------+
only showing top 5 rows

+----+---------+---------+
|open|    close| adjusted|
+----+---------+---------+
| 0.0|     51.0|7326300.0|
| 0.0|49.150002|1025200.0|
| 0.0|52.009998| 269900.0|
| 0.0|   55.423| 602800.0|
| 0.0|55.650002|1589600.0|
+----+---------+---------+
only showing top 5 rows



In [12]:
#FILTER

from pyspark.sql.functions import col, lit

data.filter((col('high') >= lit('50')) & (col('low') <= lit('60'))).show(5)


+---+------+----+---------+---------+---------+------+---------+----------+------+-------------+--------------------+
|_c0|symbol|open|     high|      low|    close|volume| adjusted|market.cap|sector|     industry|            exchange|
+---+------+----+---------+---------+---------+------+---------+----------+------+-------------+--------------------+
|  1|   TXG| 0.0|     54.0|     58.0|     51.0|     0|7326300.0|     52.75|$9.31B|Capital Goods|Biotechnology: La...|
|  2|   TXG| 0.0|    52.75|   54.355|49.150002|     0|1025200.0|     52.27|$9.31B|Capital Goods|Biotechnology: La...|
|  3|   TXG| 0.0|52.450001|     56.0|52.009998|     0| 269900.0| 55.200001|$9.31B|Capital Goods|Biotechnology: La...|
| 10|   TXG| 0.0|54.459999|55.880001|   52.563|     0| 261200.0| 52.759998|$9.31B|Capital Goods|Biotechnology: La...|
| 11|   TXG| 0.0|52.779999|53.689999|46.619999|     0| 596300.0| 49.990002|$9.31B|Capital Goods|Biotechnology: La...|
+---+------+----+---------+---------+---------+------+--

In [13]:
# WHEN

data.select("open", "close", f.when(data.adjusted >= 596300.0, 1).otherwise(0)).show(5)

NameError: name 'f' is not defined

In [ ]:
# GROUP BY

data.select(["industry", "open", "close", "adjusted"]).groupBy("industry").mean().show()

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("City analysis").getOrCreate()

df = spark.read.csv("./worldcities.csv", header=True, inferSchema=True)
df.describe().show()

+-------+----------------+--------------+------------------+------------------+-----------+-----+-----+----------+-------+------------------+--------------------+
|summary|            city|    city_ascii|               lat|               lng|    country| iso2| iso3|admin_name|capital|        population|                  id|
+-------+----------------+--------------+------------------+------------------+-----------+-----+-----+----------+-------+------------------+--------------------+
|  count|           47868|         47867|             47868|             47868|      47868|47868|47868|     47671|  13023|             47656|               47868|
|   mean|             NaN|           NaN| 25.53451267652722|16.331352542408315|       null| null| null|       NaN|   null|108922.70083515192|1.4490736349635456E9|
| stddev|            null|          null|23.000405144792392|  70.2985804390263|       null| null| null|       NaN|   null|  685789.550104917|2.6112003807384118E8|
|    min|        A Cor